In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [3]:
# 1. Load Data
df = pd.read_csv('mental_health_score.csv')

In [4]:
# 2. Drop Duplicate Rows
df = df.drop_duplicates()

# 3. Fix Invalid Data Anomalies
# Physical activity hours cannot be negative; clip lower bound at 0
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0.0)

In [8]:
df['Country'].value_counts()[df['Country'].value_counts() >= 50]

Country
Other          2363
India           389
USA             354
Canada          230
Australia       198
UK              185
Germany         136
Mexico           94
Turkey           94
France           87
Spain            83
Ireland          81
Japan            77
Denmark          77
Switzerland      73
Nepal            72
Italy            67
Russia           66
Sri Lanka        59
Maldives         57
Bangladesh       53
Pakistan         53
Poland           50
Name: count, dtype: int64

In [22]:
# 4. Consolidate High-Cardinality Categorical Columns
# Group infrequent countries (less than 1% representation) into 'Other'
top_countries = df['Country'].value_counts()[df['Country'].value_counts() >= 50].index
df['Country'] = df['Country'].apply(lambda x: x if x in top_countries[:10] else 'Other')
top_countries
df['Country'].value_counts()

Country
Other        3231
India         389
USA           354
Canada        230
Australia     198
UK            185
Germany       136
Mexico         94
Turkey         94
France         87
Name: count, dtype: int64

In [23]:

# 5. Define Feature Sets & Target
X = df.drop(columns=['Mental_Health_Score'])
y = df['Mental_Health_Score']




In [24]:
# Import XGBoost (or LightGBM)
from xgboost import XGBRegressor

In [27]:

# 1. Define Feature Names & Category Orders
numeric_features = [
    'Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks', 
    'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night'
]

academic_order = ['High School', 'Undergraduate', 'Graduate']
stress_order = ['Low', 'Medium', 'High', 'Very High']

nominal_features = ['Gender', 'Country', 'Most_Used_Platform', 'Purpose_Of_Use']

# 2. Construct Feature Transformers
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

academic_transformer = Pipeline(steps=[
    ('ordinal_academic', OrdinalEncoder(categories=[academic_order]))
])

stress_transformer = Pipeline(steps=[
    ('ordinal_stress', OrdinalEncoder(categories=[stress_order]))
])

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='infrequent_if_exist'))
])

# 3. Combine into a Single ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('acad', academic_transformer, ['Academic_Level']),
        ('stress', stress_transformer, ['Stress_Level']),
        ('nom', nominal_transformer, nominal_features)
    ]
)

# 4. Build End-to-End Pipeline with Model
# Swap XGBRegressor with LGBMRegressor() if using LightGBM
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    ))
])

# 5. Fit & Predict Directly on Raw Data Splitted Sets
# Assuming X and y are already extracted from your dataframe
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the entire pipeline (preprocessing + XGBoost fitting)
full_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [26]:
# Predict continuous scores
y_pred = full_pipeline.predict(X_test)

# 1. Regression Metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# 2. Percentage Accuracy Within Error Margin (e.g., within +- 0.5 points)
tolerance = 0.5
accuracy_within_margin = np.mean(np.abs(y_test - y_pred) <= tolerance) * 100

print(f"XGBoost Regression Evaluation:")
print(f"  R² Score                   : {r2:.4f} ({r2 * 100:.2f}% variance explained)")
print(f"  RMSE                       : {rmse:.4f}")
print(f"  MAE                        : {mae:.4f}")
print(f"  Accuracy (within ±{tolerance}) : {accuracy_within_margin:.2f}%")

XGBoost Regression Evaluation:
  R² Score                   : 0.8255 (82.55% variance explained)
  RMSE                       : 0.5582
  MAE                        : 0.4290
  Accuracy (within ±0.5) : 65.20%


In [36]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate

In [37]:
# 1. Build End-to-End Pipeline with Linear Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 2. Define 5-Fold Cross-Validation Splitter
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Define Metrics to Evaluate Across Folds
scoring_metrics = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

# 4. Perform 5-Fold Cross-Validation on Training Data (X_train)
cv_results = cross_validate(
    lr_pipeline, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring_metrics,
    return_train_score=True
)

# Extract Mean Scores Across the 5 Folds
cv_r2_training = cv_results['train_r2'].mean()
cv_r2_validation = cv_results['test_r2'].mean()
cv_mae_validation = -cv_results['test_mae'].mean()  # Convert back from negative MAE

# 5. Fit the Model on Full X_train and Evaluate on Untouched X_test (Final Benchmark)
lr_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [38]:

lr_preds_test = lr_pipeline.predict(X_test)

holdout_r2 = r2_score(y_test, lr_preds_test)
holdout_mae = mean_absolute_error(y_test, lr_preds_test)
holdout_rmse = np.sqrt(mean_squared_error(y_test, lr_preds_test))

# 6. Print Comprehensive Results
print("=" * 60)
print("  LINEAR REGRESSION — 5-FOLD CV & HOLDOUT RESULTS")
print("=" * 60)
print(f"5-Fold CV Train R² (Mean)     : {cv_r2_training:.4f} ({cv_r2_training * 100:.2f}%)")
print(f"5-Fold CV Val R² (Mean)       : {cv_r2_validation:.4f} ({cv_r2_validation * 100:.2f}%)")
print(f"5-Fold CV Val MAE (Mean)      : {cv_mae_validation:.4f}")
print("-" * 60)
print(f"Holdout Test R² Score        : {holdout_r2:.4f} ({holdout_r2 * 100:.2f}%)")
print(f"Holdout Test MAE              : {holdout_mae:.4f}")
print(f"Holdout Test RMSE             : {holdout_rmse:.4f}")
print("=" * 60)

  LINEAR REGRESSION — 5-FOLD CV & HOLDOUT RESULTS
5-Fold CV Train R² (Mean)     : 0.7271 (72.71%)
5-Fold CV Val R² (Mean)       : 0.7216 (72.16%)
5-Fold CV Val MAE (Mean)      : 0.5262
------------------------------------------------------------
Holdout Test R² Score        : 0.7436 (74.36%)
Holdout Test MAE              : 0.5328
Holdout Test RMSE             : 0.6766


In [31]:

from sklearn.ensemble import RandomForestRegressor

In [32]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('random forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [33]:
rf_preds          = rf_pipeline.predict(X_test)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing  = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

Accuracy of Training 0.9825052515748018
Accuracy of Testing 0.8892585688096389
MAE : 0.3290296666666667


In [40]:
import joblib


# ── 1. SAVE THE TRAINED PIPELINE LOCALLY ─────────────────────────────────────
# Assuming 'full_pipeline' or 'lr_pipeline' is already fitted on your data
model_filename = "mental_health_random_forest_pipeline.pkl"

joblib.dump(rf_pipeline, model_filename)
print(f"✅ Model pipeline saved locally to: {model_filename}")

✅ Model pipeline saved locally to: mental_health_random_forest_pipeline.pkl
